# 作业 02：根据三个线索，自己完成训练

请参考作业 01，独立编写完整的 PyTorch 训练流程，让下面的预测函数尽量拟合给定数据。

本题只给出三个线索。参数如何创建、如何计算梯度以及如何更新，都由你完成。可以查阅作业 01，但请根据本题的预测函数调整代码。

## 线索 1：预测函数

这次有三个待学习参数 $w_1,w_2,w_3$：

$$\hat y=w_1x+w_2x^2+w_3.$$

请根据这个公式，自己编写计算预测值的代码。输入整个 `x` 张量时，应得到每个样本对应的预测值。三个待学习参数也由你创建。请保留这个预测公式。

其中，$w_1$ 是一次项的系数，$w_2$ 是二次项的系数，$w_3$ 是常数项。三个参数在训练中都需要被调整。

## 线索 2：损失函数

每个样本的损失仍然是预测误差的绝对值：

$$L_i=|y_i-\hat y_i|.$$

本次有多个样本，请将所有样本的损失取平均，得到用于训练的一个数：

$$L=\frac{1}{N}\sum_{i=1}^{N}|y_i-\hat y_i|.$$

这里 $N$ 是样本数。请自己用 PyTorch 写出这个损失，不要换成平方误差。

## 线索 3：两个数据张量

`x[i]` 和 `y[i]` 是同一个样本的输入和目标。训练时保持这些数据不变，不要直接填写你猜测的参数作为答案。

In [1]:
import torch

# 输入数据与对应的目标数据，均为浮点数
x = torch.tensor([-2.0, -1.5, -1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0])
y = torch.tensor([-3.0, -1.375, 0.0, 1.125, 2.0, 2.625, 3.0, 3.125, 3.0])

## 你的作答区

请在下面编写完整训练代码，可以自行增加单元格。训练结束后，展示学到的三个参数、每个样本的预测值，以及最终的平均绝对误差。最终展示的预测值和损失应使用训练结束后的参数计算。

再用文字说明：你对作业 01 的训练代码做了哪些修改？你如何判断这次训练有效？不要求参数或误差恰好等于某个数，也不要求每一轮损失都下降。

In [2]:
# 三个待学习参数，初始化为 0；requires_grad=True 让 PyTorch 跟踪它们的梯度
w1 = torch.tensor(0.0, requires_grad=True)
w2 = torch.tensor(0.0, requires_grad=True)
w3 = torch.tensor(0.0, requires_grad=True)

# SGD 优化器，把三个参数一起交给它更新
optimizer = torch.optim.SGD([w1, w2, w3], lr=0.01)

epochs = 3000
history = []

for epoch in range(1, epochs + 1):
    # 预测函数：y_hat = w1*x + w2*x^2 + w3，一次喂进整个 x 张量
    y_pred = w1 * x + w2 * x ** 2 + w3
    # 损失：所有样本绝对误差的平均
    loss = torch.abs(y - y_pred).mean()

    optimizer.zero_grad()   # 清空上一轮残留的梯度
    loss.backward()         # 反向传播，计算 w1/w2/w3 各自的梯度
    optimizer.step()        # 按梯度更新三个参数

    # 用 no_grad 记录这一轮结束时的损失，不额外构建求导图
    with torch.no_grad():
        current_loss = torch.abs(y - (w1 * x + w2 * x ** 2 + w3)).mean().item()
    history.append((epoch, w1.item(), w2.item(), w3.item(), current_loss))

# 打印训练过程中几个检查点
print("轮数      w1         w2         w3         平均绝对误差")
for i in [0, 1, 9, 49, 199, 999, epochs - 1]:
    e, a, b, c, l = history[i]
    print(f"{e:4d}  {a:9.6f}  {b:9.6f}  {c:9.6f}  {l:.6f}")

# 用训练结束后的参数重新计算最终预测和损失（题目要求）
with torch.no_grad():
    y_final = w1 * x + w2 * x ** 2 + w3
    final_loss = torch.abs(y - y_final).mean()

print()
print("学到的参数：")
print(f"  w1 = {w1.item():.6f}")
print(f"  w2 = {w2.item():.6f}")
print(f"  w3 = {w3.item():.6f}")
print()
print("每个样本的输入、目标、预测与绝对误差：")
print("   x       y         y_pred     |误差|")
for xi, yi, pi in zip(x.tolist(), y.tolist(), y_final.tolist()):
    print(f"{xi:6.1f}  {yi:8.3f}  {pi:9.6f}  {abs(yi - pi):.6f}")
print()
print(f"最终平均绝对误差：{final_loss.item():.6f}")


轮数      w1         w2         w3         平均绝对误差
   1   0.008889   0.001667   0.004444  2.129043
   2   0.016667   0.004444   0.010000  2.119136
  10   0.081111   0.024444   0.052222  2.040000
  50   0.398889   0.128889   0.267778  1.644074
 200   1.251110  -0.106666   0.932223  0.679260
1000   1.499999  -0.498889   1.992217  0.005932
3000   1.499999  -0.498889   1.992217  0.005932

学到的参数：
  w1 = 1.499999
  w2 = -0.498889
  w3 = 1.992217

每个样本的输入、目标、预测与绝对误差：
   x       y         y_pred     |误差|
  -2.0    -3.000  -3.003339  0.003339
  -1.5    -1.375  -1.380283  0.005283
  -1.0     0.000  -0.006672  0.006672
  -0.5     1.125   1.117495  0.007505
   0.0     2.000   1.992217  0.007783
   0.5     2.625   2.617494  0.007506
   1.0     3.000   2.993327  0.006673
   1.5     3.125   3.119715  0.005285
   2.0     3.000   2.996658  0.003342

最终平均绝对误差：0.005932


### 我的观察与说明

**对作业 01 的训练代码做了哪些修改：**

1. 参数从 1 个 `w` 变成 3 个（`w1, w2, w3`），优化器的参数列表也相应改成 `[w1, w2, w3]`。
2. 预测公式从 `x * w ** 2` 改成 `w1 * x + w2 * x ** 2 + w3`，匹配本题的二次多项式模型。
3. 输入 `x`、目标 `y` 从单个标量换成了长度为 9 的张量；损失从单个绝对值 `torch.abs(y - y_pred)` 改成对所有样本取平均 `.mean()`，得到题目要求的一个数。
4. 学习率策略做了调整：作业 01 用的是 `base_lr / epoch` 的衰减。本题我先沿用同样的衰减试了一下，但因为有 9 个样本、3 个参数，学习率衰减到第 10 轮时已经只剩 0.01，参数还没走到位就基本停止了，损失停在 1.3 左右没收敛。所以改成固定学习率 `lr=0.01`、多跑 3000 轮，让参数有足够时间慢慢靠近最优解。

**如何判断这次训练有效：**

- 平均绝对误差从初始的约 2.13 一路降到 0.006 左右，说明预测值在不断逼近目标值。
- 训练到约 500 轮之后损失不再明显下降，参数也基本稳定，说明已经收敛。
- 学到的 `w1 ≈ 1.50`、`w2 ≈ -0.50`、`w3 ≈ 1.99`，每个样本的预测值与目标值之间的误差都在 0.01 以内，拟合良好。
- 因为损失是绝对值，在精确解附近不可导，SGD 会在最优点附近小幅震荡，所以误差不会严格为 0，这是正常现象。


## 提交

重启内核并从上到下运行全部单元格，确认代码可以完整运行。保存包含代码、输出和文字回答的本文件，再提交到自己的 GitHub 仓库。